In [1]:
from pathlib import Path

ROOT = Path("..").resolve()

DATA_DIR = ROOT / "data" / "raw" / "ml-1m"

paths = {
    "ratings": DATA_DIR / "ratings.dat",
    "movies":  DATA_DIR / "movies.dat",
    "users":   DATA_DIR / "users.dat",
}

print("ROOT =", ROOT)
print("DATA_DIR =", DATA_DIR)
for name, p in paths.items():
    if p.exists():
        print(f"{name:7s}: OK | size = {p.stat().st_size/1024/1024:.2f} MB | path = {p}")
    else:
        print(f"{name:7s}: NOT FOUND | expected path = {p}")


ROOT = C:\Users\User\plum-ml1m-repro
DATA_DIR = C:\Users\User\plum-ml1m-repro\data\raw\ml-1m
ratings: OK | size = 23.45 MB | path = C:\Users\User\plum-ml1m-repro\data\raw\ml-1m\ratings.dat
movies : OK | size = 0.16 MB | path = C:\Users\User\plum-ml1m-repro\data\raw\ml-1m\movies.dat
users  : OK | size = 0.13 MB | path = C:\Users\User\plum-ml1m-repro\data\raw\ml-1m\users.dat


In [13]:
import pandas as pd

ratings = pd.read_csv(
    paths["ratings"],
    sep="::",
    engine="python",          
    names=["user_id", "movie_id", "rating", "timestamp"],
    encoding="latin-1",
)

print("ratings shape:", ratings.shape)

display(ratings.head())

print("unique ratings:", sorted(ratings["rating"].unique()))
print("n_users:", ratings["user_id"].nunique(), "n_movies:", ratings["movie_id"].nunique())
print("timestamp min/max:", ratings["timestamp"].min(), ratings["timestamp"].max())


ratings shape: (1000209, 4)


,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


unique ratings: [1, 2, 3, 4, 5]
n_users: 6040 n_movies: 3706
timestamp min/max: 956703932 1046454590


In [3]:
movies = pd.read_csv(
    paths["movies"],
    sep="::",
    engine="python",
    names=["movie_id", "title", "genres"],
    encoding="latin-1",
)

print("movies shape:", movies.shape)
display(movies.head())


print("Example title:", movies.loc[0, "title"])
print("Example genres:", movies.loc[0, "genres"])


movies shape: (3883, 3)


,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


Example title: Toy Story (1995)
Example genres: Animation|Children's|Comedy


In [4]:
users = pd.read_csv(
    paths["users"],
    sep="::",
    engine="python",
    names=["user_id", "gender", "age", "occupation", "zip"],
    encoding="latin-1",
)

print("users shape:", users.shape)
display(users.head())
print("gender values:", users["gender"].unique())
print("age values:", sorted(users["age"].unique())[:10], "... total:", users["age"].nunique())

users shape: (6040, 5)


,user_id,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


gender values: ['F' 'M']
age values: [1, 18, 25, 35, 45, 50, 56] ... total: 7


In [5]:
PROCESSED_DIR = ROOT / "data" / "processed"
SPLITS_DIR = PROCESSED_DIR / "splits"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

print("PROCESSED_DIR =", PROCESSED_DIR)
print("SPLITS_DIR =", SPLITS_DIR)

PROCESSED_DIR = C:\Users\User\plum-ml1m-repro\data\processed
SPLITS_DIR = C:\Users\User\plum-ml1m-repro\data\processed\splits


In [6]:
import numpy as np

# Создадим отображения user_id -> user_idx и movie_id -> item_idx
unique_users = np.sort(ratings["user_id"].unique())
unique_items = np.sort(ratings["movie_id"].unique())

user2idx = {u: i for i, u in enumerate(unique_users)}
item2idx = {m: i for i, m in enumerate(unique_items)}

ratings_re = ratings.copy()
ratings_re["user_idx"] = ratings_re["user_id"].map(user2idx)
ratings_re["item_idx"] = ratings_re["movie_id"].map(item2idx)

print("Reindexed ratings shape:", ratings_re.shape)
print("user_idx min/max:", ratings_re["user_idx"].min(), ratings_re["user_idx"].max())
print("item_idx min/max:", ratings_re["item_idx"].min(), ratings_re["item_idx"].max())

Reindexed ratings shape: (1000209, 6)
user_idx min/max: 0 6039
item_idx min/max: 0 3705


In [7]:
min_history = 5

df = ratings_re.sort_values(["user_idx", "timestamp"]).copy()
lens = df.groupby("user_idx").size()

keep_users = lens[lens >= min_history].index
df = df[df["user_idx"].isin(keep_users)].copy()

df["pos"] = df.groupby("user_idx").cumcount()
df["len"] = df.groupby("user_idx")["item_idx"].transform("size")

train = df[df["pos"] < df["len"] - 2].copy()
val   = df[df["pos"] == df["len"] - 2].copy()
test  = df[df["pos"] == df["len"] - 1].copy()

print("Users kept:", df["user_idx"].nunique(), "out of", ratings_re["user_idx"].nunique())
print("Train/Val/Test sizes:", len(train), len(val), len(test))


print("val rows per user min/max:", val.groupby("user_idx").size().min(), val.groupby("user_idx").size().max())
print("test rows per user min/max:", test.groupby("user_idx").size().min(), test.groupby("user_idx").size().max())

Users kept: 6040 out of 6040
Train/Val/Test sizes: 988129 6040 6040
val rows per user min/max: 1 1
test rows per user min/max: 1 1


In [8]:
# Сохраняем reindexed рейтинги и сплиты
ratings_re.to_parquet(PROCESSED_DIR / "ratings_reindexed.parquet", index=False)
train.to_parquet(SPLITS_DIR / "train.parquet", index=False)
val.to_parquet(SPLITS_DIR / "val.parquet", index=False)
test.to_parquet(SPLITS_DIR / "test.parquet", index=False)

# Сохраним movies с item_idx (важно для контентных фич)
movies_re = movies.copy()
movies_re["item_idx"] = movies_re["movie_id"].map(item2idx)

# В movies есть фильмы, которые могли не встречаться в ratings -> item_idx станет NaN.
# Оставим только те фильмы, что реально есть в рейтингах.
movies_re = movies_re.dropna(subset=["item_idx"]).copy()
movies_re["item_idx"] = movies_re["item_idx"].astype(int)

movies_re.to_parquet(PROCESSED_DIR / "movies_reindexed.parquet", index=False)

print("Saved:")
print(" -", PROCESSED_DIR / "ratings_reindexed.parquet")
print(" -", SPLITS_DIR / "train.parquet")
print(" -", SPLITS_DIR / "val.parquet")
print(" -", SPLITS_DIR / "test.parquet")
print(" -", PROCESSED_DIR / "movies_reindexed.parquet")

Saved:
 - C:\Users\User\plum-ml1m-repro\data\processed\ratings_reindexed.parquet
 - C:\Users\User\plum-ml1m-repro\data\processed\splits\train.parquet
 - C:\Users\User\plum-ml1m-repro\data\processed\splits\val.parquet
 - C:\Users\User\plum-ml1m-repro\data\processed\splits\test.parquet
 - C:\Users\User\plum-ml1m-repro\data\processed\movies_reindexed.parquet


In [9]:
# считаем популярность айтемов в train
popular_items = train["item_idx"].value_counts().index.to_numpy(dtype=np.int64)

print("Num popular items:", len(popular_items))
print("Top-10 item_idx:", popular_items[:10].tolist())

Num popular items: 3704
Top-10 item_idx: [2651, 1106, 253, 1120, 466, 575, 1848, 1178, 2374, 579]


In [10]:
def recall_at_k(recommended: np.ndarray, true_item: int, k: int) -> float:
    return 1.0 if true_item in recommended[:k] else 0.0

Ks = [1, 5, 10, 20]
results = {k: [] for k in Ks}

# test содержит ровно 1 таргет на пользователя
for row in test.itertuples(index=False):
    true_item = row.item_idx
    for k in Ks:
        rec = popular_items[:k]
        results[k].append(recall_at_k(rec, true_item, k))

for k in Ks:
    print(f"PopRec Recall@{k}: {float(np.mean(results[k])):.4f}")


PopRec Recall@1: 0.0030
PopRec Recall@5: 0.0078
PopRec Recall@10: 0.0162
PopRec Recall@20: 0.0402
